<a href="https://colab.research.google.com/github/cout-angela/projectP_imaging/blob/main/group_p_tv_template.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computational Imaging - Group P
# Motion Deblur and Denoising: Degradation Pipeline + TV Regularization

**Corso:** Computational Imaging  
**Studenti:** [Nome Studente 1], [Nome Studente 2]  
**Dataset:** LSUN Church  
**Task:** Motion Deblur + Denoising come problema inverso  

---

## Struttura del notebook

Questo notebook copre i seguenti passi:

1. Installazione delle dipendenze e configurazione dell'ambiente
2. Download e preprocessing del dataset LSUN Church
3. Costruzione della pipeline di degradazione (motion blur + rumore)
4. Metodo variazionale: regularizzazione con Total Variation (TV)
5. Valutazione quantitativa (PSNR, SSIM) e visualizzazione

**Nota:** Tutti i metodi successivi (end-to-end, generativo, ibrido) dovranno usare **esattamente gli stessi input degradati** generati nella Sezione 3.

---

## Background teorico

Il problema di motion deblur e denoising e' formulato come **problema inverso**:

$$y^\delta = K * x + e$$

dove:
- $x \in \mathbb{R}^{H \times W}$ e' l'immagine originale (ground truth)
- $K$ e' il kernel di motion blur (PSF - Point Spread Function)
- $e \sim \mathcal{N}(0, \sigma^2 I)$ e' rumore gaussiano additivo
- $y^\delta$ e' l'osservazione degradata

Il metodo TV risolve il seguente problema di minimizzazione:

$$\min_{x \geq 0} \|K * x - y^\delta\|_2^2 + \lambda \, \text{TV}(x)$$

con la discretizzazione isotropica della Total Variation:

$$\text{TV}(x) = \sum_{i,j} \sqrt{(x_{i+1,j} - x_{i,j})^2 + (x_{i,j+1} - x_{i,j})^2}$$

La TV e' particolarmente adatta a immagini con **gradienti sparsi** (strutture piecewise-constant), preservando i bordi meglio della regolarizzazione di Tikhonov.


---
## 1. Setup dell'ambiente

In [ ]:
# Verifica della disponibilita' della GPU
import torch
print(f'CUDA disponibile: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Installazione delle librerie necessarie
# scikit-image: metriche PSNR/SSIM e utilita' per immagini
# lmdb: formato del dataset LSUN
# Pillow, matplotlib, numpy: standard

!pip install scikit-image lmdb tqdm --quiet

In [ ]:
import os
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
import scipy.ndimage

# Seed per riproducibilita'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Utilizzo device: {DEVICE}')

# Parametri globali del progetto (NON modificare qui sotto)
IMG_SIZE    = 256
KERNEL_SIZE = 9
NOISE_LEVELS = [0.005, 0.01, 0.05, 0.1]

---
## 2. Dataset LSUN Church

Le istruzioni ufficiali per il download sono disponibili su: https://github.com/fyu/lsun

Il dataset e' in formato LMDB. La cella seguente mostra come scaricarlo e come costruire un Dataset PyTorch a partire da esso.

**Struttura attesa dopo il download:**
```
./data/
    church_outdoor_train_lmdb/
    church_outdoor_val_lmdb/
```

In [ ]:
# Download del dataset LSUN Church tramite lo script ufficiale
# Eseguire solo la prima volta; richiede ~3 GB di spazio.

!git clone https://github.com/fyu/lsun.git --quiet
!mkdir -p data

# TODO: scegliere la categoria corretta (church_outdoor) e avviare il download.
# Lo script download.py di fyu/lsun accetta come argomento la categoria.
# Documentarsi sulla sintassi esatta prima di eseguire.

# !python lsun/download.py -c church_outdoor -o data/

In [ ]:
import lmdb
import io

class LSUNChurchDataset(Dataset):
    """
    Dataset PyTorch per LSUN Church in formato LMDB.

    Parametri
    ----------
    lmdb_path : str
        Percorso alla cartella .lmdb del dataset.
    transform : callable, opzionale
        Trasformazioni da applicare alle immagini.
    max_samples : int, opzionale
        Se specificato, limita il dataset ai primi N campioni.
        Utile per sviluppo e debug.
    """

    def __init__(self, lmdb_path, transform=None, max_samples=None):
        self.lmdb_path = lmdb_path
        self.transform = transform

        # Apertura read-only del database LMDB
        env = lmdb.open(lmdb_path, readonly=True, lock=False, readahead=False)
        with env.begin() as txn:
            self.length = txn.stat()['entries']
        env.close()

        if max_samples is not None:
            self.length = min(self.length, max_samples)

        # Lista delle chiavi (lazy: aperta al primo accesso)
        self._env  = None
        self._keys = None

    def _open_env(self):
        if self._env is None:
            self._env = lmdb.open(
                self.lmdb_path, readonly=True, lock=False, readahead=False
            )
            with self._env.begin() as txn:
                cursor = txn.cursor()
                self._keys = [key for key, _ in cursor.iternext_dup(keys=True)]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        self._open_env()
        with self._env.begin() as txn:
            img_bytes = txn.get(self._keys[idx])
        img = Image.open(io.BytesIO(img_bytes)).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img


# --- Preprocessing pipeline ---
# GIUSTIFICAZIONE SCELTE:
#   - Resize a 256x256: richiesto dalla traccia del progetto.
#   - CenterCrop prima del resize: rimuove bordi con artefatti JPEG frequenti
#     nel dataset LSUN, mantenendo il soggetto centrale.
#   - ToTensor: converte da PIL [0,255] a float32 [0.0, 1.0].
#   - Normalize con media 0.5 e std 0.5: porta i valori in [-1, 1].
#     ATTENZIONE: per le metriche PSNR/SSIM bisogna riportare in [0,1]
#     con la trasformazione inversa: x_01 = (x + 1) / 2.

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(256),
    transforms.ToTensor(),                        # [0, 1]
    transforms.Normalize(mean=[0.5, 0.5, 0.5],
                         std=[0.5, 0.5, 0.5]),    # [-1, 1]
])

def denormalize(tensor):
    """Riporta un tensore da [-1,1] a [0,1]."""
    return (tensor + 1.0) / 2.0


# --- Splitting del dataset ---
# TODO: definire le proporzioni dello split train/val/test.
#       Giustificare la scelta nella presentazione orale.
#       Verificare che non ci siano sovrapposizioni tra i set.
#
# Esempio (da completare con i path reali):
#
# TRAIN_LMDB = 'data/church_outdoor_train_lmdb'
# VAL_LMDB   = 'data/church_outdoor_val_lmdb'
#
# train_dataset = LSUNChurchDataset(TRAIN_LMDB, transform=preprocess, max_samples=5000)
# val_dataset   = LSUNChurchDataset(VAL_LMDB,   transform=preprocess, max_samples=500)
#
# Per il test set, estrarre un sottoinsieme separato dal train o usare
# l'intero val LMDB, documentando la scelta.

print('Dataset class definita. Completare i TODO sopra prima di procedere.')

In [ ]:
# TODO: caricare effettivamente il dataset una volta completato il download
# e verificare visivamente alcune immagini di esempio.

# Esempio di verifica visiva:
# loader = DataLoader(val_dataset, batch_size=4, shuffle=True)
# batch  = next(iter(loader))  # shape: (4, 3, 256, 256)
# batch_01 = denormalize(batch)  # riporta in [0,1] per la visualizzazione

# fig, axes = plt.subplots(1, 4, figsize=(14, 4))
# for i, ax in enumerate(axes):
#     img_np = batch_01[i].permute(1, 2, 0).numpy()
#     ax.imshow(img_np)
#     ax.axis('off')
#     ax.set_title(f'Campione {i+1}')
# plt.suptitle('Esempi dal dataset LSUN Church (preprocessed)')
# plt.tight_layout()
# plt.show()

print('TODO: completare questa cella dopo il download del dataset.')

---
## 3. Pipeline di degradazione

La degradazione segue il modello:

$$y^\delta = K * x + e, \qquad e \sim \mathcal{N}(0, \sigma^2 I)$$

I parametri fissati dalla traccia sono:
- `kernel_size = 9`
- `motion_angle`: a scelta degli studenti (da documentare e giustificare)
- `noise_level` $\sigma \in \{0.005, 0.01, 0.05, 0.1\}$

**Importante:** la stessa coppia `(immagine degradata, ground truth)` deve essere usata da tutti i metodi confrontati.

### Costruzione del kernel di motion blur

Un kernel di motion blur di lunghezza $L$ e angolo $\theta$ e' una linea di 1s in direzione $\theta$, normalizzata a somma 1.

In [ ]:
def create_motion_blur_kernel(kernel_size: int, angle_deg: float) -> np.ndarray:
    """
    Genera un kernel di motion blur di dimensione kernel_size x kernel_size
    con direzione angle_deg (in gradi).

    Il kernel e' una linea di 1s orientata nella direzione data, normalizzata
    a somma 1 in modo che la luminosita' media dell'immagine sia preservata.

    Parametri
    ----------
    kernel_size : int
        Dimensione del kernel quadrato (es. 9 -> kernel 9x9).
    angle_deg : float
        Angolo del motion blur in gradi (0 = orizzontale, 90 = verticale).

    Ritorna
    -------
    kernel : np.ndarray, shape (kernel_size, kernel_size), float32, somma=1
    """
    kernel = np.zeros((kernel_size, kernel_size), dtype=np.float32)
    center = kernel_size // 2

    # Disegna la linea ruotando le coordinate
    angle_rad = np.deg2rad(angle_deg)
    for i in range(kernel_size):
        offset = i - center
        x = int(round(center + offset * np.cos(angle_rad)))
        y = int(round(center + offset * np.sin(angle_rad)))
        if 0 <= x < kernel_size and 0 <= y < kernel_size:
            kernel[y, x] = 1.0

    # Normalizzazione: la somma del kernel deve essere 1
    total = kernel.sum()
    if total > 0:
        kernel /= total
    else:
        raise ValueError(f'Kernel vuoto con angle_deg={angle_deg}, kernel_size={kernel_size}')

    return kernel


# TODO: scegliere l'angolo del motion blur e documentare la scelta.
#       Considerare un angolo che produca un blur visivamente significativo
#       sulle chiese del dataset (es. orizzontale, diagonale).
MOTION_ANGLE = None  # TODO: sostituire con un float, es. 45.0

if MOTION_ANGLE is not None:
    kernel = create_motion_blur_kernel(KERNEL_SIZE, MOTION_ANGLE)
    plt.figure(figsize=(3, 3))
    plt.imshow(kernel, cmap='hot', interpolation='nearest')
    plt.title(f'Motion blur kernel\nsize={KERNEL_SIZE}, angle={MOTION_ANGLE} deg')
    plt.colorbar()
    plt.tight_layout()
    plt.show()
    print(f'Kernel shape: {kernel.shape}, somma: {kernel.sum():.4f}')
else:
    print('TODO: definire MOTION_ANGLE.')

In [ ]:
def apply_motion_blur(image_tensor: torch.Tensor, kernel: np.ndarray) -> torch.Tensor:
    """
    Applica il kernel di motion blur a un batch di immagini usando convoluzione 2D.

    La convoluzione e' separata per canale (groups=C) in modo da applicare
    lo stesso kernel PSF a ciascun canale R, G, B indipendentemente,
    coerentemente con un modello di blur spazio-invariante e canale-indipendente.

    Parametri
    ----------
    image_tensor : torch.Tensor, shape (B, C, H, W) o (C, H, W)
        Immagini in input in [0, 1] o [-1, 1].
    kernel : np.ndarray, shape (k, k)
        Kernel di motion blur gia' normalizzato.

    Ritorna
    -------
    blurred : torch.Tensor, stessa shape dell'input
    """
    if image_tensor.dim() == 3:
        image_tensor = image_tensor.unsqueeze(0)  # aggiunge batch dim
        squeeze = True
    else:
        squeeze = False

    B, C, H, W = image_tensor.shape
    k = kernel.shape[0]
    pad = k // 2

    # Converte il kernel in tensore: shape (C, 1, k, k) per grouped conv
    kernel_t = torch.tensor(kernel, dtype=torch.float32)
    kernel_t = kernel_t.unsqueeze(0).unsqueeze(0)  # (1, 1, k, k)
    kernel_t = kernel_t.repeat(C, 1, 1, 1)         # (C, 1, k, k)
    kernel_t = kernel_t.to(image_tensor.device)

    blurred = F.conv2d(image_tensor, kernel_t, padding=pad, groups=C)

    if squeeze:
        blurred = blurred.squeeze(0)
    return blurred


def add_gaussian_noise(image_tensor: torch.Tensor, sigma: float) -> torch.Tensor:
    """
    Aggiunge rumore gaussiano additivo white con deviazione standard sigma.

    Parametri
    ----------
    image_tensor : torch.Tensor
        Immagine o batch di immagini.
    sigma : float
        Livello di rumore (deviazione standard). I valori del progetto sono
        0.005, 0.01, 0.05, 0.1 (relativi all'intervallo [0,1]).

    Ritorna
    -------
    noisy : torch.Tensor, stessa shape dell'input, con rumore aggiunto
    """
    noise = torch.randn_like(image_tensor) * sigma
    return image_tensor + noise


def degrade_image(image_tensor: torch.Tensor,
                  kernel: np.ndarray,
                  sigma: float) -> torch.Tensor:
    """
    Applica la pipeline completa di degradazione: blur -> rumore.

    IMPORTANTE: questa funzione deve essere chiamata con lo stesso kernel
    e lo stesso seed di numpy/torch per tutti i metodi confrontati,
    in modo da garantire che tutti i metodi ricevano esattamente lo stesso input.

    Parametri
    ----------
    image_tensor : torch.Tensor, shape (C, H, W), valori in [-1, 1]
    kernel : np.ndarray
    sigma : float

    Ritorna
    -------
    degraded : torch.Tensor, shape (C, H, W)
    """
    blurred = apply_motion_blur(image_tensor, kernel)
    degraded = add_gaussian_noise(blurred, sigma)
    return degraded


print('Funzioni di degradazione definite.')

In [ ]:
# Test visivo della pipeline di degradazione su un'immagine singola.
# Eseguire questa cella solo dopo aver completato il TODO sul dataset.

# TODO: sostituire 'sample_image' con un'immagine reale dal dataset.
#       La cella mostra il confronto tra ground truth, blurred e degraded
#       per tutti i livelli di rumore richiesti dalla traccia.

if MOTION_ANGLE is not None:
    kernel = create_motion_blur_kernel(KERNEL_SIZE, MOTION_ANGLE)

    # TODO: sostituire con: sample_image = val_dataset[0]  (shape: C,H,W in [-1,1])
    sample_image = None  # placeholder

    if sample_image is not None:
        sample_01 = denormalize(sample_image)  # [0,1] per visualizzazione

        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        ax = axes.ravel()

        ax[0].imshow(sample_01.permute(1,2,0).numpy())
        ax[0].set_title('Ground Truth')
        ax[0].axis('off')

        blurred_only = apply_motion_blur(sample_image, kernel)
        ax[1].imshow(denormalize(blurred_only).permute(1,2,0).numpy().clip(0,1))
        ax[1].set_title(f'Solo Blur (kernel={KERNEL_SIZE}, angle={MOTION_ANGLE})')
        ax[1].axis('off')

        for i, sigma in enumerate(NOISE_LEVELS):
            degraded = degrade_image(sample_image, kernel, sigma)
            degraded_01 = denormalize(degraded).permute(1,2,0).numpy().clip(0,1)
            ax[i+2].imshow(degraded_01)
            ax[i+2].set_title(f'Blur + Rumore sigma={sigma}')
            ax[i+2].axis('off')

        plt.suptitle('Pipeline di degradazione')
        plt.tight_layout()
        plt.show()
    else:
        print('TODO: caricare sample_image dal dataset.')
else:
    print('TODO: definire MOTION_ANGLE prima di eseguire questa cella.')

### Salvataggio degli input degradati

E' fondamentale salvare su disco il subset di immagini degradate che verra' usato da tutti i metodi. In questo modo si garantisce la riproducibilita' e la confrontabilita' dei risultati.

In [ ]:
# TODO: generare e salvare il test set degradato.
#
# Struttura consigliata su disco (Google Drive o cartella Colab):
#
#   degraded_data/
#       sigma_0.005/
#           degraded_0000.pt, degraded_0001.pt, ...
#       sigma_0.01/
#           ...
#       sigma_0.05/
#           ...
#       sigma_0.1/
#           ...
#       ground_truth/
#           gt_0000.pt, gt_0001.pt, ...
#
# Ogni file .pt contiene un tensore (C, H, W) in [-1, 1].
# Usare torch.save() e torch.load() per la serializzazione.
#
# Numero di immagini di test suggerito: 100-500 (da documentare).

# Esempio di struttura (completare con il codice reale):
# OUTPUT_DIR = '/content/drive/MyDrive/group_p_data'  # oppure path locale
# NUM_TEST   = 200
#
# for sigma in NOISE_LEVELS:
#     os.makedirs(f'{OUTPUT_DIR}/sigma_{sigma}', exist_ok=True)
# os.makedirs(f'{OUTPUT_DIR}/ground_truth', exist_ok=True)
#
# for idx in tqdm(range(NUM_TEST)):
#     gt = test_dataset[idx]          # (C, H, W) in [-1, 1]
#     torch.save(gt, f'{OUTPUT_DIR}/ground_truth/gt_{idx:04d}.pt')
#     for sigma in NOISE_LEVELS:
#         torch.manual_seed(SEED + idx)   # seed per riproducibilita' per-immagine
#         deg = degrade_image(gt, kernel, sigma)
#         torch.save(deg, f'{OUTPUT_DIR}/sigma_{sigma}/degraded_{idx:04d}.pt')

print('TODO: completare la generazione e il salvataggio degli input degradati.')

---
## 4. Metodo variazionale: Total Variation (TV) Regularization

Il problema da risolvere e':

$$\min_{x} \|K * x - y^\delta\|_2^2 + \lambda \, \text{TV}(x)$$

Poiche' TV e' **convessa ma non differenziabile**, si ricorre a metodi di ottimizzazione prossimale.

### Approcci implementativi

Due strade principali (scegliere e documentare):

1. **Split Bregman / ADMM**: decompone il problema in sottoproblemi chiusi in forma. Efficiente e ampiamente usato in letteratura.
2. **Gradient Descent su TV smoothed**: sostituisce TV con $\text{TV}_\epsilon(x) = \sum_{i,j} \sqrt{(\nabla x)^2 + \epsilon^2}$ per renderla differenziabile, poi usa gradient descent.

Il template implementa la versione con **TV smoothed + gradient descent** (piu' semplice da implementare da zero) e lascia come TODO l'implementazione di ADMM per chi vuole andare piu' in profondita'.

### Calcolo efficiente del termine di data fidelity

Il prodotto matrice-vettore $K * x$ e' una convoluzione, quindi puo' essere calcolato nel dominio di Fourier in modo efficiente:

$$\|K * x - y^\delta\|_2^2 \Leftrightarrow \|\hat{K} \odot \hat{x} - \hat{y}^\delta\|_2^2$$

dove $\hat{\cdot}$ denota la DFT 2D.

In [ ]:
def compute_tv_isotropic(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Calcola la Total Variation isotropica (smooth) di un'immagine.

    TV(x) = sum_{i,j} sqrt( (x_{i+1,j} - x_{i,j})^2 + (x_{i,j+1} - x_{i,j})^2 + eps )

    Il termine eps > 0 rende la TV differenziabile ovunque (smoothed TV),
    a costo di una lieve approssimazione della TV esatta.

    Parametri
    ----------
    x   : torch.Tensor, shape (C, H, W) o (B, C, H, W)
    eps : float, piccola costante per la differenziabilita'

    Ritorna
    -------
    tv_value : torch.Tensor, scalare
    """
    # Differenze orizzontali e verticali (forward differences)
    diff_h = x[..., 1:, :] - x[..., :-1, :]   # shape (..., H-1, W)
    diff_v = x[..., :, 1:] - x[..., :, :-1]   # shape (..., H, W-1)

    # Per sommare elemento per elemento, ritagliamo alla dimensione comune
    diff_h = diff_h[..., :, :-1]  # (..., H-1, W-1)
    diff_v = diff_v[..., :-1, :]  # (..., H-1, W-1)

    tv = torch.sqrt(diff_h ** 2 + diff_v ** 2 + eps)
    return tv.sum()


def tv_gradient(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """
    Calcola il gradiente della TV smoothed rispetto a x usando autograd di PyTorch.

    In alternativa e' possibile calcolare il gradiente analiticamente tramite
    l'operatore divergenza discreto; questo approccio usa autograd per semplicita'.

    Parametri
    ----------
    x   : torch.Tensor, shape (C, H, W), requires_grad=True
    eps : float

    Ritorna
    -------
    grad : torch.Tensor, shape (C, H, W)
    """
    x_copy = x.detach().clone().requires_grad_(True)
    tv_val = compute_tv_isotropic(x_copy, eps)
    tv_val.backward()
    return x_copy.grad.detach()


def data_fidelity_gradient(x: torch.Tensor,
                           y_degraded: torch.Tensor,
                           kernel: np.ndarray) -> torch.Tensor:
    """
    Calcola il gradiente del termine di data fidelity:

        grad_x  ||K*x - y||^2  =  2 * K^T * (K*x - y)

    dove K^T e' la convoluzione con il kernel trasposto (kernel ruotato di 180 gradi),
    che corrisponde al gradiente della convoluzione rispetto all'input.

    Per boundary conditions cicliche puo' essere calcolato nel dominio di Fourier
    (vedi commento TODO sotto).

    Parametri
    ----------
    x          : torch.Tensor, shape (C, H, W)
    y_degraded : torch.Tensor, shape (C, H, W)
    kernel     : np.ndarray, shape (k, k)

    Ritorna
    -------
    grad : torch.Tensor, shape (C, H, W)
    """
    Kx   = apply_motion_blur(x, kernel)              # K*x
    residual = Kx - y_degraded                       # K*x - y

    # K^T e' la convoluzione con il kernel ruotato di 180 gradi
    kernel_T = np.rot90(kernel, 2).copy()
    grad = 2.0 * apply_motion_blur(residual, kernel_T)

    # TODO (opzionale, per maggiore efficienza):
    # Implementare il gradiente nel dominio di Fourier usando torch.fft.fft2
    # e la relazione:  F(K^T * r) = conj(F(K)) * F(r)
    # Questo e' equivalente ma piu' efficiente per kernel grandi.

    return grad


print('Componenti TV definite.')

In [ ]:
def tv_gradient_descent(
    y_degraded: torch.Tensor,
    kernel: np.ndarray,
    lam: float,
    n_iter: int,
    step_size: float,
    eps_tv: float = 1e-8,
    x_init: torch.Tensor = None,
    verbose: bool = True,
    gt: torch.Tensor = None
) -> tuple:
    """
    Risolve il problema di TV-regularized deblur tramite gradient descent:

        min_x ||K*x - y||^2 + lambda * TV(x)

    Algoritmo:
        x_{t+1} = x_t - step_size * (grad_fidelity(x_t) + lambda * grad_TV(x_t))

    Nota: per convergenza garantita usare uno step size <= 1 / L dove L e'
    la costante di Lipschitz del gradiente del termine di fidelity.
    In pratica, si sceglie step_size empiricamente con line search o
    si usa un metodo come FISTA per convergenza piu' rapida.

    Parametri
    ----------
    y_degraded : torch.Tensor, shape (C, H, W), immagine degradata in [-1,1]
    kernel     : np.ndarray, PSF normalizzata
    lam        : float, parametro di regolarizzazione lambda
    n_iter     : int, numero di iterazioni
    step_size  : float, passo del gradient descent
    eps_tv     : float, termine di smoothing della TV
    x_init     : torch.Tensor, inizializzazione (default: y_degraded)
    verbose    : bool, se True stampa la loss ogni 50 iterazioni
    gt         : torch.Tensor, ground truth per monitorare PSNR durante le iterazioni

    Ritorna
    -------
    x_rec      : torch.Tensor, shape (C, H, W), immagine ricostruita
    loss_history : list of float
    psnr_history : list of float (vuota se gt=None)
    """
    # Inizializzazione
    if x_init is None:
        x = y_degraded.clone().to(DEVICE)
    else:
        x = x_init.clone().to(DEVICE)

    y = y_degraded.to(DEVICE)
    loss_history = []
    psnr_history = []

    for t in range(n_iter):
        # Gradiente del termine di data fidelity
        grad_fid = data_fidelity_gradient(x, y, kernel).to(DEVICE)

        # Gradiente della TV smoothed
        grad_tv  = tv_gradient(x, eps_tv).to(DEVICE)

        # Aggiornamento gradient descent
        x = x - step_size * (grad_fid + lam * grad_tv)

        # Calcolo della loss
        fid_loss = torch.sum((apply_motion_blur(x, kernel) - y) ** 2).item()
        tv_loss  = compute_tv_isotropic(x, eps_tv).item()
        total    = fid_loss + lam * tv_loss
        loss_history.append(total)

        # Monitoraggio PSNR (solo se ground truth disponibile)
        if gt is not None:
            x01 = denormalize(x.cpu()).permute(1,2,0).numpy().clip(0,1)
            g01 = denormalize(gt.cpu()).permute(1,2,0).numpy().clip(0,1)
            p   = psnr(g01, x01, data_range=1.0)
            psnr_history.append(p)

        if verbose and (t % 50 == 0 or t == n_iter - 1):
            msg = f'Iter {t:4d}/{n_iter} | Loss={total:.4f} (fid={fid_loss:.4f}, tv={tv_loss:.4f})'
            if psnr_history:
                msg += f' | PSNR={psnr_history[-1]:.2f} dB'
            print(msg)

    return x.detach(), loss_history, psnr_history


print('Solver TV definito.')

In [ ]:
# --- Scelta euristica dei parametri TV ---
#
# I parametri principali da scegliere per la TV sono:
#
#   lambda    : parametro di regolarizzazione.
#               Troppo piccolo -> poco effetto, risultato ancora rumoroso/blurred.
#               Troppo grande  -> immagine eccessivamente smooth, perdita di dettagli.
#               Ricerca consigliata: griglia logaritmica, es. [1e-4, 1e-3, 1e-2, 1e-1].
#
#   step_size : passo del gradient descent.
#               Deve essere sufficientemente piccolo per la convergenza.
#               Consiglio: partire da 1e-4 e dimezzare se la loss non scende.
#
#   n_iter    : numero di iterazioni.
#               Monitorare la loss: fermarsi quando essa si stabilizza.
#               Tipicamente 200-500 iterazioni sono sufficienti.
#
# TODO: riempire i dizionari con i valori scelti per ogni livello di rumore.
#       Documentare il processo di scelta nella presentazione.

# Parametri per ogni livello di rumore (da compilare)
TV_PARAMS = {
    0.005: {'lam': None, 'step_size': None, 'n_iter': None},  # TODO
    0.01:  {'lam': None, 'step_size': None, 'n_iter': None},  # TODO
    0.05:  {'lam': None, 'step_size': None, 'n_iter': None},  # TODO
    0.1:   {'lam': None, 'step_size': None, 'n_iter': None},  # TODO
}

# Suggerimento: usare la cella seguente per un grid search veloce su
# una singola immagine di validazione prima di girare l'intera valutazione.

print('Parametri TV da completare.')

In [ ]:
# Grid search euristico di lambda su un'immagine di validazione.
# Eseguire questa cella per ogni livello di rumore.
#
# TODO: sostituire val_image e val_degraded con dati reali.

SIGMA_TO_TUNE = 0.01  # Cambiare per ogni livello di rumore
LAMBDA_GRID   = [1e-4, 5e-4, 1e-3, 5e-3, 1e-2, 5e-2, 1e-1]
STEP_SIZE     = 1e-4   # TODO: regolare se la loss non converge
N_ITER_SEARCH = 200    # Iterazioni ridotte per la ricerca

# val_image    = val_dataset[0]   # TODO: shape (C,H,W) in [-1,1]
# val_degraded = degrade_image(val_image, kernel, SIGMA_TO_TUNE)

# results_grid = []
# for lam in LAMBDA_GRID:
#     x_rec, _, _ = tv_gradient_descent(
#         val_degraded, kernel,
#         lam=lam, n_iter=N_ITER_SEARCH, step_size=STEP_SIZE,
#         verbose=False, gt=val_image
#     )
#     x01 = denormalize(x_rec.cpu()).permute(1,2,0).numpy().clip(0,1)
#     g01 = denormalize(val_image).permute(1,2,0).numpy().clip(0,1)
#     p   = psnr(g01, x01, data_range=1.0)
#     s   = ssim(g01, x01, data_range=1.0, channel_axis=2)
#     results_grid.append({'lam': lam, 'psnr': p, 'ssim': s})
#     print(f'lam={lam:.0e} | PSNR={p:.2f} dB | SSIM={s:.4f}')
#
# best = max(results_grid, key=lambda r: r['psnr'])
# print(f"\nMiglior lambda: {best['lam']} (PSNR={best['psnr']:.2f} dB)")

print('TODO: eseguire il grid search dopo aver caricato il dataset.')

---
## 5. Valutazione quantitativa e visualizzazione

Per ogni livello di rumore e per ogni immagine di test, calcolare:
- **PSNR** (Peak Signal-to-Noise Ratio, dB): misura la fedelta' pixel-wise.
- **SSIM** (Structural Similarity Index): misura la similarita' strutturale percettiva.

I risultati devono essere riportati come media $\pm$ deviazione standard sul test set.

In [ ]:
def compute_metrics(gt_tensor: torch.Tensor,
                    rec_tensor: torch.Tensor) -> dict:
    """
    Calcola PSNR e SSIM tra l'immagine ground truth e quella ricostruita.

    Entrambi i tensori devono essere in [-1, 1] con shape (C, H, W).
    Vengono convertiti in [0, 1] prima del calcolo delle metriche.

    Parametri
    ----------
    gt_tensor  : torch.Tensor, ground truth
    rec_tensor : torch.Tensor, ricostruzione

    Ritorna
    -------
    metrics : dict con chiavi 'psnr' e 'ssim'
    """
    gt_np  = denormalize(gt_tensor.cpu()).permute(1,2,0).numpy().clip(0, 1)
    rec_np = denormalize(rec_tensor.cpu()).permute(1,2,0).numpy().clip(0, 1)

    p = psnr(gt_np, rec_np, data_range=1.0)
    s = ssim(gt_np, rec_np, data_range=1.0, channel_axis=2)
    return {'psnr': p, 'ssim': s}


print('Funzione metriche definita.')

In [ ]:
# Valutazione completa del metodo TV su tutto il test set.
#
# TODO: sostituire 'test_images' con il caricamento reale dei dati
#       salvati nella Sezione 3 (ground truth + degraded).

# Struttura dei risultati: dizionario sigma -> lista di metriche per immagine
tv_results = {sigma: [] for sigma in NOISE_LEVELS}

# for sigma in NOISE_LEVELS:
#     params = TV_PARAMS[sigma]
#     assert params['lam'] is not None, f'TODO: parametri TV per sigma={sigma} non definiti'
#
#     for idx in tqdm(range(NUM_TEST), desc=f'TV sigma={sigma}'):
#         gt      = torch.load(f'{OUTPUT_DIR}/ground_truth/gt_{idx:04d}.pt')
#         deg     = torch.load(f'{OUTPUT_DIR}/sigma_{sigma}/degraded_{idx:04d}.pt')
#
#         x_rec, _, _ = tv_gradient_descent(
#             deg, kernel,
#             lam=params['lam'],
#             n_iter=params['n_iter'],
#             step_size=params['step_size'],
#             verbose=False
#         )
#         metrics = compute_metrics(gt, x_rec)
#         tv_results[sigma].append(metrics)
#
# # Stampa riepilogo
# print('\n--- Risultati TV Regularization ---')
# print(f'{"Sigma":>8} | {"PSNR mean":>10} | {"PSNR std":>9} | {"SSIM mean":>10} | {"SSIM std":>9}')
# for sigma in NOISE_LEVELS:
#     psnr_vals = [m['psnr'] for m in tv_results[sigma]]
#     ssim_vals = [m['ssim'] for m in tv_results[sigma]]
#     print(f'{sigma:>8} | {np.mean(psnr_vals):>10.2f} | {np.std(psnr_vals):>9.2f} | '
#           f'{np.mean(ssim_vals):>10.4f} | {np.std(ssim_vals):>9.4f}')

print('TODO: completare la valutazione dopo aver impostato i parametri e il dataset.')

In [ ]:
# Grafico di convergenza della loss e del PSNR durante le iterazioni.
# Da eseguire su una singola immagine rappresentativa.
#
# TODO: sostituire con dati reali dopo aver definito i parametri.

# Esempio:
# SIGMA_DEMO = 0.05
# params = TV_PARAMS[SIGMA_DEMO]
#
# gt_demo   = val_dataset[0]
# deg_demo  = degrade_image(gt_demo, kernel, SIGMA_DEMO)
#
# x_rec, loss_hist, psnr_hist = tv_gradient_descent(
#     deg_demo, kernel,
#     lam=params['lam'], n_iter=params['n_iter'], step_size=params['step_size'],
#     verbose=True, gt=gt_demo
# )
#
# fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
# ax1.semilogy(loss_hist)
# ax1.set_xlabel('Iterazione')
# ax1.set_ylabel('Loss (scala log)')
# ax1.set_title('Convergenza della loss')
# ax1.grid(True)
#
# ax2.plot(psnr_hist)
# ax2.set_xlabel('Iterazione')
# ax2.set_ylabel('PSNR (dB)')
# ax2.set_title('Evoluzione del PSNR')
# ax2.grid(True)
#
# plt.suptitle(f'TV Regularization - sigma={SIGMA_DEMO}, lambda={params["lam"]}')
# plt.tight_layout()
# plt.show()

print('TODO: generare i grafici di convergenza.')

In [ ]:
# Confronto visivo: ground truth / degraded / ricostruita
# per tutti i livelli di rumore.
#
# TODO: sostituire con dati reali.

# Esempio di figura finale per la presentazione:
#
# fig, axes = plt.subplots(len(NOISE_LEVELS), 3, figsize=(12, 4*len(NOISE_LEVELS)))
# col_titles = ['Ground Truth', 'Degradata', 'TV Reconstruction']
#
# for row, sigma in enumerate(NOISE_LEVELS):
#     params  = TV_PARAMS[sigma]
#     gt_img  = val_dataset[0]
#     deg_img = degrade_image(gt_img, kernel, sigma)
#
#     x_rec, _, _ = tv_gradient_descent(
#         deg_img, kernel,
#         lam=params['lam'], n_iter=params['n_iter'], step_size=params['step_size'],
#         verbose=False
#     )
#     metrics = compute_metrics(gt_img, x_rec)
#
#     for col, (img_t, title) in enumerate(zip([gt_img, deg_img, x_rec], col_titles)):
#         img_np = denormalize(img_t.cpu()).permute(1,2,0).numpy().clip(0,1)
#         axes[row, col].imshow(img_np)
#         axes[row, col].axis('off')
#         if row == 0:
#             axes[row, col].set_title(title, fontsize=12)
#         if col == 2:
#             axes[row, col].set_xlabel(
#                 f'PSNR={metrics["psnr"]:.2f} dB | SSIM={metrics["ssim"]:.4f}',
#                 fontsize=9
#             )
#
# # Etichette righe con sigma
# for row, sigma in enumerate(NOISE_LEVELS):
#     axes[row, 0].set_ylabel(f'sigma={sigma}', fontsize=10, rotation=0,
#                             labelpad=50, va='center')
#
# plt.suptitle('TV Regularization - Confronto visivo per livello di rumore', fontsize=13)
# plt.tight_layout()
# plt.savefig('tv_visual_comparison.png', dpi=150, bbox_inches='tight')
# plt.show()

print('TODO: generare il confronto visivo.')

---
## 6. Note per i metodi successivi

I notebook per i metodi end-to-end (UNet/ViT/NAFNet), generativo (DiffPIR) e ibrido (PnP-HQS) devono:

1. Caricare gli input degradati salvati nella Sezione 3 di questo notebook, **senza rigenerarli**.
2. Usare le stesse funzioni `compute_metrics` e `denormalize` definite qui (importarle o copiarle).
3. Produrre risultati nella stessa struttura `{sigma: [{'psnr': ..., 'ssim': ...}, ...]}` per facilitare il confronto finale.

Il notebook di confronto finale raccogliera' i risultati di tutti i metodi e produrra' il comparison plot richiesto dalla traccia.

In [ ]:
# Struttura del comparison plot finale (da completare quando tutti i metodi sono pronti).
#
# METHODS = {
#     'Degraded':       degraded_results,
#     'TV':             tv_results,
#     'End-to-End':     e2e_results,    # da notebook separato
#     'DiffPIR':        diff_results,   # da notebook separato
#     'PnP-HQS':        pnp_results,    # da notebook separato
# }
#
# fig, (ax_psnr, ax_ssim) = plt.subplots(1, 2, figsize=(14, 5))
# x_pos = np.arange(len(NOISE_LEVELS))
#
# for name, results in METHODS.items():
#     psnr_means = [np.mean([m['psnr'] for m in results[s]]) for s in NOISE_LEVELS]
#     ssim_means = [np.mean([m['ssim'] for m in results[s]]) for s in NOISE_LEVELS]
#     ax_psnr.plot(x_pos, psnr_means, marker='o', label=name)
#     ax_ssim.plot(x_pos, ssim_means, marker='s', label=name)
#
# ax_psnr.set_xticks(x_pos)
# ax_psnr.set_xticklabels([str(s) for s in NOISE_LEVELS])
# ax_psnr.set_xlabel('Livello di rumore sigma')
# ax_psnr.set_ylabel('PSNR (dB)')
# ax_psnr.set_title('PSNR per metodo e livello di rumore')
# ax_psnr.legend()
# ax_psnr.grid(True)
#
# ax_ssim.set_xticks(x_pos)
# ax_ssim.set_xticklabels([str(s) for s in NOISE_LEVELS])
# ax_ssim.set_xlabel('Livello di rumore sigma')
# ax_ssim.set_ylabel('SSIM')
# ax_ssim.set_title('SSIM per metodo e livello di rumore')
# ax_ssim.legend()
# ax_ssim.grid(True)
#
# plt.suptitle('Confronto quantitativo - tutti i metodi')
# plt.tight_layout()
# plt.savefig('comparison_all_methods.png', dpi=150, bbox_inches='tight')
# plt.show()

print('Comparison plot: da completare quando tutti i metodi sono implementati.')